# Task 1: Dataset Understanding

In [1]:
# Import libraries
import pandas as pd

# Load dataset
df = pd.read_csv("customer_support_text_classification.csv")

# Display first 5 rows
print(df.head())

# ---------------------------------------------------
# Number of records
# ---------------------------------------------------
print("Number of Records:", len(df))

# ---------------------------------------------------
# Target labels/classes
# ---------------------------------------------------
print("\nTarget Classes:")
print(df["sentiment_label"].unique())

# ---------------------------------------------------
# Sample text records
# ---------------------------------------------------
print("\nSample Text Records:")
print(df["customer_message"].head())

# ---------------------------------------------------
# Average text length
# ---------------------------------------------------
df["text_length"] = df["customer_message"].apply(lambda x: len(str(x).split()))

print("\nAverage Text Length:",
      round(df["text_length"].mean(), 2), "words")

# ---------------------------------------------------
# Class distribution
# ---------------------------------------------------
print("\nClass Distribution:")
print(df["sentiment_label"].value_counts())

  ticket_id channel                                   customer_message  \
0  TKT00001    chat  I need information about the payment process. ...   
1  TKT00002   phone      I need information about the payment process.   
2  TKT00003   email  The refund process was fast and convenient. I ...   
3  TKT00004  social  My refund is still pending and this experience...   
4  TKT00005    chat   Please tell me how to update my account details.   

  sentiment_label  word_count  urgent_flag  
0         neutral          18            1  
1         neutral           7            0  
2        positive          12            0  
3        negative          15            1  
4         neutral           9            0  
Number of Records: 1500

Target Classes:
<StringArray>
['neutral', 'positive', 'negative']
Length: 3, dtype: str

Sample Text Records:
0    I need information about the payment process. ...
1        I need information about the payment process.
2    The refund process was fast and con

# Task 2: Text Preprocessing

In [5]:
# Import required libraries
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Download NLTK resources
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

# ---------------------------------------------------
# Step 1: Lowercasing + Removing Special Characters
# ---------------------------------------------------

def clean_text(text):
    text = text.lower()  # convert to lowercase
    text = re.sub(r'[^a-zA-Z\s]', '', text)  # remove symbols/numbers
    return text

df["cleaned_text"] = df["customer_message"].apply(clean_text)

# ---------------------------------------------------
# Step 2: Tokenization
# ---------------------------------------------------

df["tokens"] = df["cleaned_text"].apply(word_tokenize)

# ---------------------------------------------------
# Step 3: Remove Stopwords
# ---------------------------------------------------

stop_words = set(stopwords.words('english'))

df["filtered_tokens"] = df["tokens"].apply(
    lambda words: [word for word in words if word not in stop_words]
)

# ---------------------------------------------------
# Step 4: Convert Text to Sequences
# ---------------------------------------------------

tokenizer = Tokenizer()
tokenizer.fit_on_texts(df["filtered_tokens"])

sequences = tokenizer.texts_to_sequences(df["filtered_tokens"])

# ---------------------------------------------------
# Step 5: Padding Sequences
# ---------------------------------------------------

max_length = 20

padded_sequences = pad_sequences(
    sequences,
    maxlen=max_length,
    padding='post',
    truncating='post'
)

# ---------------------------------------------------
# Display Results
# ---------------------------------------------------

print("Original Text:\n")
print(df["customer_message"].iloc[0])

print("\nCleaned Text:\n")
print(df["cleaned_text"].iloc[0])

print("\nTokens:\n")
print(df["tokens"].iloc[0])

print("\nFiltered Tokens:\n")
print(df["filtered_tokens"].iloc[0])

print("\nPadded Sequence Shape:")
print(padded_sequences.shape)

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Original Text:

I need information about the payment process. My ticket number is 78732. Please respond as soon as possible.

Cleaned Text:

i need information about the payment process my ticket number is  please respond as soon as possible

Tokens:

['i', 'need', 'information', 'about', 'the', 'payment', 'process', 'my', 'ticket', 'number', 'is', 'please', 'respond', 'as', 'soon', 'as', 'possible']

Filtered Tokens:

['need', 'information', 'payment', 'process', 'ticket', 'number', 'please', 'respond', 'soon', 'possible']

Padded Sequence Shape:
(1500, 20)


# Task 3: Text Vectorization

In [6]:
# Import TF-IDF Vectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

# ---------------------------------------------------
# Convert Text into Numerical Vectors using TF-IDF
# ---------------------------------------------------

tfidf = TfidfVectorizer(max_features=5000)

X = tfidf.fit_transform(df["cleaned_text"])

# Display shape of vectorized data
print("Shape of TF-IDF Matrix:", X.shape)

# Display some feature names
print("\nSample Features:")
print(tfidf.get_feature_names_out()[:20])

Shape of TF-IDF Matrix: (1500, 180)

Sample Features:
['about' 'account' 'activate' 'after' 'ago' 'am' 'analytics' 'and' 'any'
 'app' 'appreciate' 'arrived' 'as' 'assigned' 'available' 'bad' 'because'
 'been' 'between' 'billing']


## Explanation
Machine learning and deep learning models cannot understand raw text directly because they work only with numbers.
Text vectorization converts words and sentences into numerical representations (vectors) so that models can process patterns, relationships, and frequencies in the data.

In this task, TF-IDF (Term Frequency–Inverse Document Frequency) is used because:
--> It converts text into numerical format
--> Gives importance to meaningful words
--> Reduces the importance of very common words
--> Works well for text classification problems like sentiment analysis
Example
Text:
"I love this product"
After vectorization:
[0.12, 0.45, 0.78, ...]
These numerical vectors are then used as input for machine learning or neural network models.

# Task 4: Baseline Model

In [11]:
# Import required libraries
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# ---------------------------------------------------
# Features and Target
# ---------------------------------------------------

X = tfidf.fit_transform(df["cleaned_text"])
y = df["sentiment_label"]

# ---------------------------------------------------
# Split Dataset
# ---------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# ---------------------------------------------------
# Build Baseline Model
# Logistic Regression + TF-IDF
# ---------------------------------------------------

model = LogisticRegression()

model.fit(X_train, y_train)

# ---------------------------------------------------
# Predictions
# ---------------------------------------------------

y_pred = model.predict(X_test)

# Save sample predictions

with open("results/sample_predictions.txt", "w") as f:
    for i in range(10):
        f.write(f"Text: {df['customer_message'].iloc[i]}\n")
        f.write(f"Predicted Label: {y_pred[i]}\n")
        f.write("-" * 50 + "\n")

print("sample_predictions.txt saved successfully")

# ---------------------------------------------------
# Evaluation
# ---------------------------------------------------

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", round(accuracy * 100, 2), "%")

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred))

# Create evaluation results dataframe
import pandas as pd

evaluation_df = pd.DataFrame({
    "Metric": ["Accuracy"],
    "Value": [accuracy]
})

# Save to CSV
evaluation_df.to_csv("results/model_evaluation.csv", index=False)

print("model_evaluation.csv saved successfully")

sample_predictions.txt saved successfully
Accuracy: 100.0 %

Classification Report:

              precision    recall  f1-score   support

    negative       1.00      1.00      1.00       109
     neutral       1.00      1.00      1.00       104
    positive       1.00      1.00      1.00        87

    accuracy                           1.00       300
   macro avg       1.00      1.00      1.00       300
weighted avg       1.00      1.00      1.00       300


Confusion Matrix:

[[109   0   0]
 [  0 104   0]
 [  0   0  87]]
model_evaluation.csv saved successfully


## Explanation
A baseline model is a simple initial model used to measure basic performance before trying advanced deep learning models.

In this task:

--> TF-IDF converts text into numerical vectors
--> Logistic Regression performs text classification
--> The dataset is split into training and testing sets
--> Model performance is evaluated using:
a) Accuracy
b) Classification Report
c) Confusion Matrix

Metrics Used
1. Accuracy --> Overall correct predictions
2. Precision --> Correct positive predictions
3. Recall --> Ability to identify actual classes
4. F1-score --> Balance between precision and recall
5. Confusion Matrix --> Shows correct and incorrect predictions for each class

# Task 5: Sequence Model or Conceptual Architecture

In [8]:
# Import required libraries
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

# ---------------------------------------------------
# Build LSTM Sequence Model
# ---------------------------------------------------

vocab_size = 5000
embedding_dim = 64
max_length = 20

model = Sequential()

# Embedding Layer
model.add(Embedding(
    input_dim=vocab_size,
    output_dim=embedding_dim,
    input_length=max_length
))

# LSTM Layer
model.add(LSTM(64))

# Output Layer
model.add(Dense(3, activation='softmax'))

# ---------------------------------------------------
# Compile Model
# ---------------------------------------------------

model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

# ---------------------------------------------------
# Display Model Summary
# ---------------------------------------------------

model.summary()

c:\Users\HP\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\embedding.py:103: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

## Model Architecture Explanation
1. Input Sequence

The input text is first converted into numerical sequences using tokenization and padding.

Example:

"I love this product"

Converted sequence:

[15, 48, 7, 102]

After padding:

[15, 48, 7, 102, 0, 0, 0, ...]

All sequences are padded to the same length (max_length = 20).

2. Embedding Layer

The embedding layer converts integer word indices into dense numerical vectors.

Embedding(input_dim=5000, output_dim=64)
5000 → vocabulary size
64 → embedding vector dimension

This layer helps the model learn semantic relationships between words.

3. Recurrent / Sequence Layer
LSTM(64)

The LSTM layer processes text sequentially and remembers important information from previous words.

Why LSTM is useful:

Handles sequential text data
Learns context and word dependencies
Reduces vanishing gradient problems compared to standard RNNs
4. Output Layer
Dense(3, activation='softmax')
3 represents the three sentiment classes:
positive
negative
neutral

Softmax activation outputs probability scores for each class.

5. Loss Function
sparse_categorical_crossentropy

Used because:

This is a multiclass classification problem
Target labels are integer encoded
6. Evaluation Metric
accuracy

Accuracy measures the percentage of correctly predicted sentiment labels.

Example:

90 correct predictions out of 100
Accuracy = 90%

## Task 6: Attention and Transformer Reflection

### 1. Why RNNs Struggle with Long-Term Dependencies

RNNs process text one word at a time and pass information from previous steps to future steps. When sequences become very long, earlier information gradually becomes weak or lost. This problem is called the vanishing gradient problem.

As a result, standard RNNs struggle to remember important words that appeared much earlier in the sequence.

---

### 2. How LSTMs Help with Memory

LSTMs (Long Short-Term Memory networks) improve RNNs by introducing memory cells and gates.

These gates control:
- What information to keep
- What information to forget
- What information to pass forward

This allows LSTMs to remember important information for longer periods and better understand sequence context.

LSTMs perform better than simple RNNs for tasks like sentiment analysis, text generation, and language translation.

---

### 3. What Attention Solves in Sequence-to-Sequence Tasks

The attention mechanism helps the model focus on the most relevant words in the input sequence while generating output.

It allows the model to:
- Give importance to relevant words
- Handle long sentences better
- Improve sequence understanding

Attention improves tasks such as machine translation, summarization, and question answering.

---

### 4. Why Transformers Are Important in Modern NLP and Generative AI

Transformers use attention mechanisms instead of sequential processing.

Advantages of transformers:
- Faster training using parallel processing
- Better handling of long-range dependencies
- Improved contextual understanding
- High performance on large datasets

Transformers are the foundation of modern NLP and Generative AI models such as:
- GPT
- BERT
- ChatGPT
- Gemini

They are widely used in chatbots, text generation, translation, summarization, and many AI applications.